In [1]:
import sys
print(sys.executable)

/opt/anaconda3/envs/docai/bin/python


In [2]:
import importlib

packages = [
    "requests",
    "dotenv",
    "langgraph",
    "pdfplumber",
    "docx",
    "PIL",
    "pydantic",
]

for pkg in packages:
    try:
        importlib.import_module(pkg)
        print(f"✅ {pkg} is installed")
    except ImportError:
        print(f"❌ {pkg} NOT installed")


✅ requests is installed
✅ dotenv is installed
✅ langgraph is installed
✅ pdfplumber is installed
✅ docx is installed
✅ PIL is installed
✅ pydantic is installed


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    print("✅ API key loaded successfully")
else:
    print("❌ API key NOT found")

✅ API key loaded successfully


In [6]:
import requests
# import os
# from dotenv import load_dotenv

# load_dotenv()
# API_KEY = os.getenv("GEMINI_API_KEY")

url = f"https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash:generateContent?key={api_key}"

payload = {
    "contents": [
        {"parts": [{"text": "Say hello in one sentence"}]}
    ]
}

response = requests.post(url, json=payload)

print(response.status_code)
print(response.json())


200
{'candidates': [{'content': {'parts': [{'text': 'Hello there!'}], 'role': 'model'}, 'finishReason': 'STOP', 'index': 0}], 'usageMetadata': {'promptTokenCount': 5, 'candidatesTokenCount': 3, 'totalTokenCount': 34, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 5}], 'thoughtsTokenCount': 26, 'serviceTier': 'standard'}, 'modelVersion': 'gemini-2.5-flash', 'responseId': 'BAy-avSuFuSljuMP_IKkwAM'}


In [7]:
from typing import TypedDict, Optional, Dict, Any, List

# defining the state that will be passed to each node, each node will be configured to modify only their respective fields
class DocumentState(TypedDict):
    # --- Input ---
    file_name: str
    file_type: Optional[str]          # pdf, docx, image
    raw_bytes: bytes                  # original uploaded file
    extracted_text: Optional[str]     # text extracted from file

    # --- Classification ---
    document_type: Optional[str]      # invoice, purchase_order, receipt, other
    classification_confidence: Optional[float]

    # --- Extraction ---
    extracted_fields: Optional[Dict[str, Any]] # date, ref no, location

    # --- Validation ---
    validation_errors: Optional[List[str]] # to be defined
    is_valid: Optional[bool]

    # --- Final Output ---
    final_output: Optional[Dict[str, Any]]


In [8]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("GEMINI_API_KEY")

def call_gemini(prompt):
    
    url = (
        "https://generativelanguage.googleapis.com/"
        f"v1/models/gemini-2.5-flash:generateContent?key={API_KEY}"
    )

    payload = {
        "contents": [
            {
                "parts": [
                    {"text": prompt}
                ]
            }
        ]
    }

    response = requests.post(url, json=payload)

    if response.status_code != 200:
        raise Exception(
            f"Gemini API error {response.status_code}: "
            f"{response.text}"
        )

    result = response.json()

    if "candidates" not in result:
        raise Exception(f"Unexpected Gemini response: {result}")

    return result["candidates"][0]["content"]["parts"][0]["text"]

In [9]:
import pdfplumber
from docx import Document
from PIL import Image
import io


def ingest_node(state: DocumentState) -> DocumentState:
    print("Running ingest node")

    file_name = state["file_name"]
    raw_bytes = state["raw_bytes"]

    # Detect file type
    if file_name.endswith(".pdf"):
        state["file_type"] = "pdf"

        with pdfplumber.open(io.BytesIO(raw_bytes)) as pdf:
            text = "\n".join(page.extract_text() or "" for page in pdf.pages)

        state["extracted_text"] = text

    elif file_name.endswith(".docx"):
        state["file_type"] = "docx"

        doc = Document(io.BytesIO(raw_bytes))
        text = "\n".join(p.text for p in doc.paragraphs)

        state["extracted_text"] = text

    elif file_name.endswith((".png", ".jpg", ".jpeg")):
        state["file_type"] = "image"

        # placeholder for OCR later
        state["extracted_text"] = "image text extraction not implemented"

    else:
        state["file_type"] = "unknown"
        state["extracted_text"] = ""

    return state

In [10]:
import json

def classify_node(state: DocumentState) -> DocumentState:
    print("Running classify node")

    text = state["extracted_text"][:5000]

    prompt = f"""
You are a document classification agent.

Classify the document into exactly one of these categories:

- invoice
- purchase_order
- receipt
- other

Return ONLY valid JSON in this exact format:

{{
    "document_type": "invoice",
    "confidence": 0.95
}}

Rules:
- document_type must be exactly one of:
  invoice, purchase_order, receipt, other
- confidence must be a number between 0 and 1.
- Do not include markdown.
- Do not include any explanation.

Document:
{text}
"""

    response = call_gemini(prompt)

    try:
        result = json.loads(response)
    except json.JSONDecodeError:
        raise ValueError(
            f"Gemini returned invalid classification JSON:\n{response}"
        )

    state["document_type"] = result["document_type"]
    state["classification_confidence"] = float(result["confidence"])

    return state

In [11]:
import json

def extract_node(state: DocumentState) -> DocumentState:
    print("Running extract node")

    doc_type = state["document_type"]
    text = state["extracted_text"][:6000]

    if doc_type == "invoice":

        prompt = f"""
You are an invoice extraction agent.

Extract the following fields from the invoice:

- invoice_number
- vendor
- customer
- invoice_date
- due_date
- subtotal
- tax
- total_amount

Return ONLY valid JSON in exactly this structure:

{{
    "invoice_number": null,
    "vendor": null,
    "customer": null,
    "invoice_date": null,
    "due_date": null,
    "subtotal": null,
    "tax": null,
    "total_amount": null
}}

Rules:
- Use null if a field cannot be found.
- Do not guess missing information.
- Numeric money fields must be numbers, without currency symbols.
- Keep dates as strings.
- Do not include markdown.
- Do not include explanations.

Document:
{text}
"""

        response = call_gemini(prompt)

        try:
            data = json.loads(response)
        except json.JSONDecodeError:
            raise ValueError(
                f"Gemini returned invalid extraction JSON:\n{response}"
            )

        state["extracted_fields"] = data

    else:
        state["extracted_fields"] = {}

    return state

In [45]:
from pydantic import BaseModel, model_validator, ValidationError
from datetime import datetime


class InvoiceSchema(BaseModel):
    invoice_number: str
    vendor: str
    customer: str
    invoice_date: str
    due_date: str
    subtotal: float
    tax: float
    total_amount: float

    @model_validator(mode="after")
    def validate_invoice(self):

        errors = []

        # -------------------------
        # 1. Required fields
        # -------------------------

        required_fields = {
            "invoice_number": self.invoice_number,
            "vendor": self.vendor,
            "customer": self.customer,
            "invoice_date": self.invoice_date,
            "due_date": self.due_date,
        }

        for field, value in required_fields.items():
            if not value or not str(value).strip():
                errors.append(f"{field} is missing")

        # -------------------------
        # 2. Amounts cannot be negative
        # -------------------------

        if self.subtotal < 0:
            errors.append("Subtotal cannot be negative")

        if self.tax < 0:
            errors.append("Tax cannot be negative")

        if self.total_amount < 0:
            errors.append("Total amount cannot be negative")

        # -------------------------
        # 3. Subtotal + tax = total
        # -------------------------

        expected_total = self.subtotal + self.tax

        if abs(expected_total - self.total_amount) > 0.01:
            errors.append(
                f"Subtotal + tax ({expected_total:.2f}) "
                f"does not equal total amount ({self.total_amount:.2f})"
            )

        # -------------------------
        # 4. Tax cannot exceed total
        # -------------------------

        if self.tax > self.total_amount:
            errors.append(
                "Tax cannot be greater than total amount"
            )

        # -------------------------
        # 5. Total cannot be less than subtotal
        # -------------------------

        if self.total_amount < self.subtotal:
            errors.append(
                "Total amount cannot be less than subtotal"
            )

        # -------------------------
        # 6. Dates must be valid
        # -------------------------

        try:
            invoice_date = datetime.strptime(
                self.invoice_date, "%d/%m/%Y"
            )
        except ValueError:
            try:
                invoice_date = datetime.strptime(
                    self.invoice_date, "%B %d, %Y"
                )
            except ValueError:
                errors.append(
                    f"Invalid invoice date: {self.invoice_date}"
                )
                invoice_date = None

        try:
            due_date = datetime.strptime(
                self.due_date, "%d/%m/%Y"
            )
        except ValueError:
            try:
                due_date = datetime.strptime(
                    self.due_date, "%B %d, %Y"
                )
            except ValueError:
                errors.append(
                    f"Invalid due date: {self.due_date}"
                )
                due_date = None

        # -------------------------
        # 7. Due date cannot be before invoice date
        # -------------------------

        if invoice_date and due_date:

            if due_date < invoice_date:
                errors.append(
                    "Due date cannot be before invoice date"
                )

        # -------------------------
        # Raise all business errors
        # -------------------------

        if errors:
            raise ValueError(" | ".join(errors))

        return self

In [46]:
def validate_node(state: DocumentState) -> DocumentState:
    print("Running validate node")

    data = state["extracted_fields"]

    try:
        InvoiceSchema(**data)

        state["is_valid"] = True
        state["validation_errors"] = []

    except ValidationError as e:

        state["is_valid"] = False

        state["validation_errors"] = [
            error["msg"]
            for error in e.errors()
        ]

    except ValueError as e:

        state["is_valid"] = False

        state["validation_errors"] = [str(e)]

    return state

In [34]:
def publish_node(state: DocumentState) -> DocumentState:
    print("Running publish node")

    state["final_output"] = {
        "document_type": state["document_type"],
        "data": state["extracted_fields"],
        "valid": state["is_valid"]
    }

    return state

In [35]:
def review_node(state: DocumentState) -> DocumentState:
    print("Running review node")

    state["final_output"] = {
        "status": "manual_review_required",
        "document_type": state["document_type"],
        "data": state["extracted_fields"],
        "validation_errors": state["validation_errors"]
    }

    return state

In [48]:
from langgraph.graph import StateGraph, END
# Create graph builder
builder = StateGraph(DocumentState)

# Add nodes
builder.add_node("ingest", ingest_node)
builder.add_node("classify", classify_node)
builder.add_node("extract", extract_node)
builder.add_node("validate", validate_node)
builder.add_node("publish", publish_node)
builder.add_node("review", review_node)

# Set entry point
builder.set_entry_point("ingest")

# Linear edges
builder.add_edge("ingest", "classify")
builder.add_edge("classify", "extract")
builder.add_edge("extract", "validate")


In [49]:
def route_after_validation(state: DocumentState):
    if state["is_valid"]:
        return "publish"
    return "review"


builder.add_conditional_edges(
    "validate",
    route_after_validation,
    {
        "publish": "publish",
        "review": "review"
    }
)

# End states
builder.add_edge("publish", END)
builder.add_edge("review", END)

In [50]:
app = builder.compile()

In [39]:
with open("wrong_tota.pdf", "rb") as f:
    pdf_bytes = f.read()

print(type(pdf_bytes))
print(len(pdf_bytes))

<class 'bytes'>
99726


In [40]:
initial_state = {
    "file_name": "wrong_tota.pdf",
    "file_type": None,
    "raw_bytes": pdf_bytes,
    "extracted_text": None,
    "document_type": None,
    "classification_confidence": None,
    "extracted_fields": None,
    "validation_errors": None,
    "is_valid": None,
    "final_output": None
}

# result = app.invoke(initial_state)

# print(result)

result = ingest_node(initial_state)

print(result["file_type"])
print(result["extracted_text"][:1000])


Running ingest node
pdf
INVOICE
DATE: 10/10/2026
INVOICE NO.:
INV-9987
Due: 31/10/2026
BILL TO SHIP TO
Saniya Saniya Jim Halpert
Dunder mifflin Dunder mifflin Schneider
Scranton Scranton NYC
123456789 123456789 567898765
DESCRIPTION QTY UNIT PRICE TOTAL
Paper 200 1 200
A6 Paper 100 2 200
SUBTOTAL 400
Remarks / Payment Instructions: DISCOUNT 100
SUBTOTAL
LESS 300
DISCOUNT
TAX RATE 10%
TOTAL TAX 30
SHIPPING/
50
HANDLING
Balance
$ 400
Due


In [21]:
print(call_gemini("What type of document is an invoice? Answer in one word."))

Financial


In [41]:
state = ingest_node(initial_state)
state = classify_node(state)
state = extract_node(state)
state = validate_node(state)

print("Valid:", state["is_valid"])
print("Errors:", state["validation_errors"])

Running ingest node
Running classify node
Running extract node
Running validate node
Valid: False
Errors: ['Input should be a valid string']


In [42]:
test_state = {
    "file_name": "sample.pdf",
    "file_type": "pdf",
    "raw_bytes": b"",
    "extracted_text": "",
    "document_type": "invoice",
    "classification_confidence": 0.95,
    "extracted_fields": {
        "invoice_number": "INV-3337",
        "vendor": "DEMO - Sliced Invoices",
        "customer": "Test Business",
        "invoice_date": "January 25, 2016",
        "due_date": "January 31, 2016",
        "subtotal": 85.0,
        "tax": 8.5,
        "total_amount": 93.5
    },
    "validation_errors": [],
    "is_valid": True,
    "final_output": None
}

test_state = publish_node(test_state)

print(test_state["final_output"])

Running publish node
{'document_type': 'invoice', 'data': {'invoice_number': 'INV-3337', 'vendor': 'DEMO - Sliced Invoices', 'customer': 'Test Business', 'invoice_date': 'January 25, 2016', 'due_date': 'January 31, 2016', 'subtotal': 85.0, 'tax': 8.5, 'total_amount': 93.5}, 'valid': True}


In [43]:
test_state["is_valid"] = False
test_state["validation_errors"] = ["Missing invoice number"]
test_state["final_output"] = None

test_state = review_node(test_state)

print(test_state["final_output"])

Running review node
{'status': 'manual_review_required', 'document_type': 'invoice', 'data': {'invoice_number': 'INV-3337', 'vendor': 'DEMO - Sliced Invoices', 'customer': 'Test Business', 'invoice_date': 'January 25, 2016', 'due_date': 'January 31, 2016', 'subtotal': 85.0, 'tax': 8.5, 'total_amount': 93.5}, 'validation_errors': ['Missing invoice number']}


In [51]:
result = app.invoke(initial_state)

print("\n--- FINAL RESULT ---")
print("Document type:", result["document_type"])
print("Confidence:", result["classification_confidence"])
print("Extracted fields:", result["extracted_fields"])
print("Valid:", result["is_valid"])
print("Validation errors:", result["validation_errors"])
print("Final output:", result["final_output"])

Running ingest node
Running classify node
Running extract node
Running validate node
Running review node

--- FINAL RESULT ---
Document type: invoice
Confidence: 0.99
Extracted fields: {'invoice_number': 'INV-9987', 'vendor': None, 'customer': 'Saniya Saniya Dunder mifflin', 'invoice_date': '10/10/2026', 'due_date': '31/10/2026', 'subtotal': 400, 'tax': 30, 'total_amount': 400}
Valid: False
Validation errors: ['Input should be a valid string']
Final output: {'status': 'manual_review_required', 'document_type': 'invoice', 'data': {'invoice_number': 'INV-9987', 'vendor': None, 'customer': 'Saniya Saniya Dunder mifflin', 'invoice_date': '10/10/2026', 'due_date': '31/10/2026', 'subtotal': 400, 'tax': 30, 'total_amount': 400}, 'validation_errors': ['Input should be a valid string']}
